# Q-learning on FrozenLake

We learn $Q(s,a)$ online with the off-policy TD update (**Q-learning**):

$$Q(s,a) \leftarrow Q(s,a) - \alpha \Big[\, Q(s,a) - \big( r + \gamma\, \max_{a'} Q(s',a') \big) \Big]$$

- The **behaviour policy** is fully exploratory: ε = 1.0 (fixed), so every action is chosen uniformly at random.
- Q-learning is **off-policy**: the $\max$ over $Q(s',\cdot)$ in the target learns the values of the *greedy* policy,
  even though the agent behaves randomly. Compare with SARSA, which uses $Q(s',a')$ with the action $a'$ actually taken.
- α decays over episodes so that Q converges. At the end, the greedy policy is $\pi^*(s) = \arg\max_a Q(s,a)$.

In [ ]:
# Uncomment if needed. `reward_schedule` requires a recent Gymnasium release (>= 1.3).
# %pip install "gymnasium>=1.3" matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import gymnasium as gym

print("Gymnasium version:", gym.__version__)

## 1. The environment

- `S` start · `F` frozen (safe) · `H` hole (episode ends, reward −1) · `G` goal (episode ends, reward +1). Every other step gives reward 0.
- States are numbered row by row: `s = row * 4 + col`, so the start is 0 and the goal is 15.
- Actions: `0 = LEFT`, `1 = DOWN`, `2 = RIGHT`, `3 = UP` (note: a different order from Example 01, and there is no STAY).
- With `is_slippery=False` every action does exactly what it says.

In [ ]:
# ------------------------------------------------------------------ #
# Environment                                                        #
# ------------------------------------------------------------------ #
env = gym.make("FrozenLake-v1",
               map_name="4x4",
               is_slippery=False,
               reward_schedule=(1, -1, 0),   # (goal, hole, step)
               render_mode="ansi")

N         = env.observation_space.n       # 16 states
N_ACTIONS = env.action_space.n            # 4 actions
GAMMA     = 0.9

LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
A_NAMES  = ["LEFT", "DOWN", "RIGHT", "UP"]
A_ARROWS = ["←",    "↓",    "→",     "↑" ]

DESC = env.unwrapped.desc.astype(str)     # 4x4 array of 'S','F','H','G'

print("Number of states :", N)
print("Number of actions:", N_ACTIONS)
env.reset()
print(env.render())

## 2. How to interact with the environment

- `s, info = env.reset()` starts a new episode and returns the initial state.
- `sp, r, terminated, truncated, info = env.step(a)` takes action `a`.
  - `terminated = True` → we fell in a hole or reached the goal.
  - `truncated = True` → the 100-step time limit was reached.

In [ ]:
s, info = env.reset(seed=0)
print("start state:", s)

for a in [DOWN, DOWN, RIGHT]:
    sp, r, terminated, truncated, info = env.step(a)
    print(f"s={s:2d}  a={A_NAMES[a]:5s} -> s'={sp:2d}  r={r:+}  terminated={terminated}  truncated={truncated}")
    s = sp

## 3. ε-greedy action selection

Same as in the SARSA notebook. Ties are broken randomly. With ε = 1.0 this function always returns a random action.

In [ ]:
# ------------------------------------------------------------------ #
# Epsilon-greedy action selection                                    #
# ------------------------------------------------------------------ #
def epsilon_greedy(Q, s, eps, rng):
    """
    With probability eps    → pick a random action (explore)
    With probability 1-eps  → pick argmax_a Q(s,a)  (exploit)
    """
    if rng.random() < eps:
        return int(rng.integers(0, N_ACTIONS))
    best_actions = np.flatnonzero(Q[s] == Q[s].max())   # all maximal actions
    return int(rng.choice(best_actions))                # break ties randomly

## 4. Q-learning

The code is the same as in Example 01. Only the interaction with the environment changes:

| Gridworld (Example 01)              | FrozenLake (Gymnasium)                                    |
|-------------------------------------|-----------------------------------------------------------|
| `s = rng.integers(0, N)`            | `s, _ = env.reset()` (always starts at **S**)             |
| `sp, r = transition(s, a)`          | `sp, r, terminated, truncated, _ = env.step(a)`           |
| `while steps < max_steps:`          | stop when `terminated` (hole/goal) or `truncated` (100 steps) |

Note the order inside the loop: first update $Q(s,a)$ with $\max_{a'} Q(s',a')$, **then** choose the next action
with the behaviour policy. The next action is not needed for the update (unlike SARSA).

✏️ **Exercise.** Complete the two lines marked `# TODO` with the Q-learning update:

$$\delta = Q(s,a) - \big(r + \gamma\, \max_{a'} Q(s',a')\big), \qquad Q(s,a) \leftarrow Q(s,a) - \alpha\,\delta$$

*Hint:* `np.max(Q[sp])` gives $\max_{a'} Q(s',a')$.

In [ ]:
# ------------------------------------------------------------------ #
# Q-learning                                                         #
# ------------------------------------------------------------------ #
def q_learning(
    env,
    n_episodes  = 10000,
    eps         = 1.0,      # behaviour policy: fully exploratory, fixed
    alpha_0     = 0.5,      # initial learning rate
    alpha_min   = 0.01,     # floor for alpha
    decay_every = 500,      # decay period (episodes)
    decay_rate  = 0.90,     # multiplicative decay factor
    seed        = 42,
):
    rng = np.random.default_rng(seed)
    Q   = np.zeros((N, N_ACTIONS))      # action-value table
    env.reset(seed=seed)

    alpha = alpha_0

    ep_returns  = []   # sum of rewards per episode (for plotting)
    ep_lengths  = []   # steps per episode

    for ep in range(n_episodes):
        if (ep%2000 == 0):
            print(f"episode: {ep} done")

        # ---- decay schedule (alpha only — eps is fixed) ------------
        if ep > 0 and ep % decay_every == 0:
            alpha = max(alpha * decay_rate, alpha_min)

        # ---- start of episode: always at 'S' (state 0) -------------
        s, _ = env.reset()
        a    = epsilon_greedy(Q, s, eps, rng) # behaviour policy (eps = 1.0, always random)

        total_r = 0
        steps   = 0

        # ---- Q-learning update loop --------------------------------
        while True:
            sp, r, terminated, truncated, _ = env.step(a)

            # Note: holes and the goal are terminal. Q is never updated there,
            # so Q[sp, :] stays 0 and np.max(Q[sp]) = 0 when sp is terminal.
            # Core Q-learning update: target uses max over Q(s',·)
            td_error  = None                     # TODO: complete (off-policy target)
            Q[s, a]   = Q[s, a] - None           # TODO: complete

            total_r += r
            steps   += 1
            s = sp
            a = epsilon_greedy(Q, s, eps, rng)        # behaviour policy (eps = 1.0, always random)

            if terminated or truncated:               # end of episode
                break

        ep_returns.append(total_r)
        ep_lengths.append(steps)

    return Q, ep_returns, ep_lengths

## 5. Run Q-learning

In [ ]:
# ------------------------------------------------------------------ #
# Run Q-learning                                                     #
# ------------------------------------------------------------------ #
print("Running Q-learning …")
Q, ep_returns, ep_lengths = q_learning(env)
print("Done!")

## 6. Results

The optimal path from S to G takes 6 steps, so with γ = 0.9 the optimal values along it are
$1,\ 0.9,\ 0.81,\ 0.73,\ 0.66,\ 0.59$ (one factor of γ per extra step). Compare with the table below.

In [ ]:
# ------------------------------------------------------------------ #
# Greedy policy                                                      #
# ------------------------------------------------------------------ #
greedy_policy = np.argmax(Q, axis=1)

print("\nLearned Q-table (max Q per state, reshaped 4x4):")
print(np.array2string(np.max(Q, axis=1).reshape(4, 4),
                      formatter={"float_kind": lambda x: f"{x:5.2f}"}))

print("\nGreedy policy (arrows, H = hole, G = goal):")
grid_arrows = np.array([A_ARROWS[a] for a in greedy_policy]).reshape(4, 4)
grid_arrows[np.isin(DESC, ["H", "G"])] = DESC[np.isin(DESC, ["H", "G"])]
for row in grid_arrows:
    print("  " + "   ".join(row))

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation policy                                               #
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 1, figsize=(4, 4))

ax = axes

# Background colour per cell type: frozen = white, hole = orange, goal = blue
cell_colors = np.zeros((4, 4))
cell_colors[DESC == "H"] = 1
cell_colors[DESC == "G"] = 2

cmap_bg = ListedColormap(["white", "#FF7F27", "#00A2E8"])
ax.imshow(cell_colors, cmap=cmap_bg, vmin=0, vmax=2, alpha=0.9)

dx = { LEFT:(-.35,0), DOWN:(0,.35), RIGHT:(.35,0), UP:(0,-.35) }
for s in range(N):
    r, c = s // 4, s % 4
    if DESC[r, c] in "HG":                      # terminal: no action
        ax.text(c, r, DESC[r, c], ha="center", va="center", fontsize=14, weight="bold")
        continue
    a = greedy_policy[s]
    ddx, ddy = dx[a]
    ax.annotate("", xy=(c+ddx, r+ddy), xytext=(c-ddx, r-ddy),
                arrowprops=dict(arrowstyle="-|>", color="green", lw=1.8))
ax.text(-0.4, -0.3, "S", fontsize=10, weight="bold")

ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_title("π*(s) = argmax Q(s,·)")

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------ #
# Q-values (per action)                                              #
# ------------------------------------------------------------------ #
print("        LEFT  DOWN RIGHT    UP")
for i in range(Q.shape[0]):
    if (i%4 == 0):
        print(31*'-')
    print(f"s_{i:02d}:", np.array2string(Q[i,:], formatter={"float_kind": lambda x: f"{x:5.2f}"}))

### Returns of the behaviour policy

These are the returns of the episodes the agent actually played, with the **random** behaviour policy (ε = 1.0).
They stay close to −1 (the agent almost always falls into a hole), and that is fine:
Q-learning is off-policy, so what it learned is the **greedy** policy shown above, not the behaviour.

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation returns                                              #
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 1, figsize=(7, 4))

ax = axes
window = 100
moving_avg = np.convolve(ep_returns, np.ones(window) / window, mode="valid")
ax.plot(ep_returns, color="tab:blue", linewidth=0.5, alpha=0.3, label="return")
ax.plot(range(window - 1, len(ep_returns)), moving_avg, color="tab:blue",
        linewidth=2.0, label=f"moving average ({window} episodes)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return  (+1 goal, -1 hole, 0 timeout)")
ax.set_title("Return (behaviour policy, ε = 1.0)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Behaviour policy reached the goal in {np.mean(np.array(ep_returns) == 1):.1%} of the episodes")

## 7. Save the policy

We save the whole Q-table (not only the arrows): the script `03_run_policy_4x4.py` picks the greedy action $\arg\max_a Q(s,a)$ itself.

Then run the policy from a terminal, inside the `Lesson08` folder:

```
python 03_run_policy_4x4.py                 # 1 episode
python 03_run_policy_4x4.py --episodes 3    # 3 episodes
```

The script opens a window with the frozen lake (it needs `pip install "gymnasium[toy-text]"`) and prints whether the agent reached the goal.

In [ ]:
# ------------------------------------------------------------------ #
# Save the learned Q-table (used by 03_run_policy_4x4.py)            #
# ------------------------------------------------------------------ #
np.save("policy.npy", Q)
print("Saved policy.npy with shape", Q.shape)

## 8. Try this

1. **SARSA vs Q-learning.** Run the SARSA notebook from Lesson07 with ε = 1.0 fixed (`eps_0 = eps_min = 1.0`). Does the greedy policy of SARSA still reach the goal? Why not? (Hint: SARSA learns the values of the policy it *follows*; Q-learning learns the values of the *greedy* policy.)
2. **Less exploration.** Call `q_learning(env, eps=0.1)`. Does the greedy policy still reach the goal? Look at the returns plot now.
3. **Fewer episodes.** How few episodes (`n_episodes`) are enough to get the optimal policy?
4. **Slippery ice.** Set `is_slippery=True`. How many episodes do you need now? Does the greedy policy still take the shortest path?